## Modelado Avanzado de Series Temporales: Dengue en Cali

### Universidad Autonoma de Occidente
### Maestria en Inteligencia Artificial y Ciencias de Datos

**Dataset:** registros individuales de dengue agregados semanalmente  
**Objetivo:** comparar modelos estadisticos, suavizacion exponencial y machine learning para pronosticar casos semanales de dengue en Cali, manteniendo continuidad estricta con la fase 1.

---

### Mapa del Notebook

| Parte | Contenido |
|---|---|
| 1 | Instalacion, librerias y parametros del experimento |
| 2 | Carga de datos y reconstruccion de la serie semanal |
| 3 | Split temporal e imputacion sin fuga de informacion |
| 4 | EDA: ACF/PACF y lectura de estacionalidad |
| 5 | Estacionariedad y ruido blanco |
| 6 | Baselines heredados de fase 1 |
| 7 | Modelos ARIMA/SARIMA y suavizacion exponencial |
| 8 | Feature engineering con MLForecast |
| 9 | Modelos ML y optimizacion temporal de hiperparametros |
| 10 | Cross-validation y benchmark integral |
| 11 | Pronosticos sobre test, residuos y seleccion final |

La estructura sigue el flujo del Workshop S6: explorar ACF/PACF, diagnosticar estacionariedad, construir features, entrenar modelos, comparar por CV/test y diagnosticar residuos. La diferencia principal es que aqui se conserva el split e imputacion de fase 1 antes del EDA para evitar fuga de informacion y trabajar con una serie semanal completa.


---
## PARTE 1: INSTALACION Y CONFIGURACION

Se recomienda ejecutar este notebook con el kernel `datascience-venv` ubicado en `~/.virtualenvs/datascience-venv/`. Si se ejecuta en un entorno nuevo, instalar previamente:

```python
%pip install pandas numpy plotly matplotlib scipy statsmodels statsforecast utilsforecast mlforecast scikit-learn lightgbm
```

La configuracion replica los parametros principales de fase 1: frecuencia semanal `W-MON`, identificador `dengue_cali`, corte temporal `2020-01-01` y estacionalidad cuatrienal de 208 semanas para el baseline heredado.

In [19]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import jarque_bera
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tools import add_constant

from statsforecast import StatsForecast
from statsforecast.models import (
    ARIMA,
    AutoARIMA,
    AutoETS,
    DynamicOptimizedTheta,
    Naive,
    RandomWalkWithDrift,
    SeasonalNaive,
    WindowAverage,
)

from mlforecast import MLForecast
from utilsforecast.losses import mae, rmse, mape
from utilsforecast.evaluation import evaluate
from utilsforecast.feature_engineering import fourier
from mlforecast.lag_transforms import RollingMean, RollingStd, RollingMin, RollingMax
from mlforecast.target_transforms import Differences

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
np.random.seed(42)

RUTA_DATOS = Path("proyecto_1/datos_dengue_202604282143.csv")
if not RUTA_DATOS.exists():
    RUTA_DATOS = Path("../proyecto_1/datos_dengue_202604282143.csv")

UNIQUE_ID = "dengue_cali"
COLUMNA_FECHA = "fec_not"
FRECUENCIA = "W-MON"
FECHA_CORTE = "2020-01-01"
ESTACIONALIDAD = 52*4
SEED = 42

MODELOS_BASELINE = ["Naive", "SeasonalNaive", "WindowAverage", "RWD"]
MODELOS_FASE2_STATS = [f"AutoARIMA_{ESTACIONALIDAD}", "SARIMA_manual", f"AutoETS_{ESTACIONALIDAD}", f"Theta_{ESTACIONALIDAD}"]
MODELOS_FASE2_ML = ["ridge", "lasso", "random_forest", "lightgbm"]

print("Librerias y parametros cargados")
print(f"Ruta de datos: {RUTA_DATOS}")

Librerias y parametros cargados
Ruta de datos: ../proyecto_1/datos_dengue_202604282143.csv


---
## PARTE 2: CARGA Y RECONSTRUCCION DE LA SERIE

Se reutilizan las funciones de fase 1 para mantener exactamente el mismo criterio de agregacion semanal y formato Nixtla. La serie queda en columnas `unique_id`, `ds`, `y`.

In [20]:
def leer_datos_dengue(ruta_archivo):
    """Carga el archivo original de dengue."""
    datos = pd.read_csv(ruta_archivo)
    print("Primeras filas del dataset original:")
    display(datos.head())
    print("\nColumnas disponibles:")
    print(datos.columns.tolist())
    return datos

def construir_serie_semanal_nixtla(datos, columna_fecha="fec_not", unique_id="dengue_cali"):
    """
    Convierte el dataset diario/individual en una serie semanal con formato Nixtla:
    unique_id, ds, y.
    """
    datos = datos.copy()
    datos[columna_fecha] = pd.to_datetime(datos[columna_fecha], errors="coerce")
    datos = datos.dropna(subset=[columna_fecha])

    calendario_iso = datos[columna_fecha].dt.isocalendar()
    datos["anio"] = calendario_iso.year.astype(int)
    datos["semana"] = calendario_iso.week.astype(int)

    # Lunes de cada semana ISO. Este formato evita ambigüedades entre año calendario y año ISO.
    datos["ds"] = pd.to_datetime(
        datos["anio"].astype(str) + "-W" + datos["semana"].astype(str).str.zfill(2) + "-1",
        format="%G-W%V-%u"
    )

    serie = (
        datos.groupby("ds")
        .size()
        .reset_index(name="y")
        .sort_values("ds")
        .reset_index(drop=True)
    )

    serie.insert(0, "unique_id", unique_id)
    serie = serie[["unique_id", "ds", "y"]]
    return serie

def completar_calendario_semanal(serie, frecuencia="W-MON", unique_id="dengue_cali"):
    """Crea el calendario semanal completo y deja NaN donde falten semanas."""
    rango_completo = pd.date_range(serie.ds.min(), serie.ds.max(), freq=frecuencia)
    calendario = pd.DataFrame({"ds": rango_completo, "unique_id": unique_id})
    serie_completa = calendario.merge(serie, on=["unique_id", "ds"], how="left")
    return serie_completa[["unique_id", "ds", "y"]]

def diagnosticar_faltantes(serie, frecuencia="W-MON"):
    """Revisa continuidad temporal y valores faltantes en y."""
    fechas_esperadas = pd.date_range(serie.ds.min(), serie.ds.max(), freq=frecuencia)
    fechas_faltantes = fechas_esperadas.difference(serie.ds)

    print("=" * 55)
    print("  DIAGNÓSTICO DE VALORES FALTANTES")
    print("=" * 55)
    print(f"  Observaciones esperadas: {len(fechas_esperadas)}")
    print(f"  Observaciones presentes: {len(serie)}")
    print(f"  Fechas faltantes:        {len(fechas_faltantes)}")
    print(f"  NaN en columna y:        {serie.y.isna().sum()}")

    if len(fechas_faltantes) == 0 and serie.y.isna().sum() == 0:
        print("\n  ✅ Serie completa — sin valores faltantes")
    else:
        print(f"\n  ⚠️  Fechas faltantes: {fechas_faltantes.tolist()}")

    return fechas_faltantes


datos_crudos = leer_datos_dengue(RUTA_DATOS)

df = construir_serie_semanal_nixtla(
    datos=datos_crudos,
    columna_fecha=COLUMNA_FECHA,
    unique_id=UNIQUE_ID,
)

print("\nSerie semanal original:")
display(df.head())
display(df.tail())

fechas_faltantes = diagnosticar_faltantes(df, frecuencia=FRECUENCIA)
df = completar_calendario_semanal(df, frecuencia=FRECUENCIA, unique_id=UNIQUE_ID)

print("\nDespues de completar calendario:")
diagnosticar_faltantes(df, frecuencia=FRECUENCIA)

Primeras filas del dataset original:


,fec_not,semana
0,2010-12-03,48
1,2010-02-25,7
2,2010-01-16,1
3,2010-05-24,19
4,2010-03-26,11



Columnas disponibles:
['fec_not', 'semana']

Serie semanal original:


,unique_id,ds,y
0,dengue_cali,2009-12-28,12
1,dengue_cali,2010-01-04,142
2,dengue_cali,2010-01-11,210
3,dengue_cali,2010-01-18,253
4,dengue_cali,2010-01-25,345


,unique_id,ds,y
736,dengue_cali,2024-02-05,4
737,dengue_cali,2024-02-12,7
738,dengue_cali,2024-02-19,6
739,dengue_cali,2024-03-04,5
740,dengue_cali,2024-03-25,1


  DIAGNÓSTICO DE VALORES FALTANTES
  Observaciones esperadas: 744
  Observaciones presentes: 741
  Fechas faltantes:        3
  NaN en columna y:        0

  ⚠️  Fechas faltantes: [Timestamp('2024-02-26 00:00:00'), Timestamp('2024-03-11 00:00:00'), Timestamp('2024-03-18 00:00:00')]

Despues de completar calendario:
  DIAGNÓSTICO DE VALORES FALTANTES
  Observaciones esperadas: 744
  Observaciones presentes: 744
  Fechas faltantes:        0
  NaN en columna y:        3

  ⚠️  Fechas faltantes: []


DatetimeIndex([], dtype='datetime64[ns]', freq='W-MON')

---
## PARTE 3: SPLIT TEMPORAL E IMPUTACION SIN FUGA

El split temporal es el mismo de fase 1. La imputacion se calcula con medias por semana epidemiologica usando solo train, y luego se aplica a train/test. Esta restriccion evita fuga de informacion desde el periodo 2020-2024.

In [21]:
def separar_train_test(serie, fecha_corte):
    """Hace split temporal. NO se hace split aleatorio."""
    fecha_corte = pd.to_datetime(fecha_corte)
    train = serie[serie.ds < fecha_corte].copy()
    test = serie[serie.ds >= fecha_corte].copy()
    return train, test

def graficar_split(train, test, fecha_corte):
    """Grafica el conjunto de entrenamiento y prueba."""
    proporcion_train = len(train) / (len(train) + len(test)) * 100
    fecha_corte = pd.to_datetime(fecha_corte)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=train.ds, y=train.y, mode="lines+markers",
        name="Train", line=dict(color="#2196F3", width=2), marker=dict(size=3)
    ))
    fig.add_trace(go.Scatter(
        x=test.ds, y=test.y, mode="lines+markers",
        name="Test", line=dict(color="#FF5722", width=2), marker=dict(size=3)
    ))
    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
        line=dict(dash="dash", color="gray", width=1.5)
    )
    fig.add_annotation(
        x=fecha_corte, y=1.02, yref="paper",
        text="Corte", showarrow=False,
        font=dict(size=10, color="gray"), xanchor="left"
    )
    fig.update_layout(
        title=f"Train-Test Split Temporal — {proporcion_train:.0f}% / {100 - proporcion_train:.0f}%",
        xaxis_title="Fecha", yaxis_title="Casos",
        height=380, template="plotly_white"
    )
    fig.show()

def calcular_medias_estacionales(train):
    """Calcula la media por semana del año usando únicamente el train."""
    train_aux = train.copy()
    train_aux["semana_anio"] = train_aux.ds.dt.isocalendar().week.astype(int)
    medias = train_aux.groupby("semana_anio")["y"].mean().to_dict()
    media_global = train_aux["y"].mean()
    return medias, media_global

def imputar_por_media_estacional(datos, medias_estacionales, media_global):
    """Imputa NaN usando la media estacional calculada previamente."""
    datos = datos.copy()
    datos["semana_anio"] = datos.ds.dt.isocalendar().week.astype(int)

    def imputar_fila(fila):
        if pd.isna(fila["y"]):
            return medias_estacionales.get(fila["semana_anio"], media_global)
        return fila["y"]

    datos["y"] = datos.apply(imputar_fila, axis=1)
    datos = datos.drop(columns=["semana_anio"])
    return datos


train, test = separar_train_test(df, FECHA_CORTE)

print(f"TRAIN: {len(train)} observaciones ({train.ds.min().date()} -> {train.ds.max().date()})")
print(f"TEST:  {len(test)} observaciones ({test.ds.min().date()} -> {test.ds.max().date()})")
graficar_split(train, test, FECHA_CORTE)

medias_estacionales, media_global_train = calcular_medias_estacionales(train)
train = imputar_por_media_estacional(train, medias_estacionales, media_global_train)
test = imputar_por_media_estacional(test, medias_estacionales, media_global_train)
df = pd.concat([train, test], ignore_index=True).sort_values("ds").reset_index(drop=True)

print(f"NaN train: {train.y.isna().sum()} | NaN test: {test.y.isna().sum()} | NaN total: {df.y.isna().sum()}")
display(df.tail())

TRAIN: 523 observaciones (2009-12-28 -> 2019-12-30)
TEST:  221 observaciones (2020-01-06 -> 2024-03-25)


NaN train: 0 | NaN test: 0 | NaN total: 0


,unique_id,ds,y
739,dengue_cali,2024-02-26,121.8
740,dengue_cali,2024-03-04,5.0
741,dengue_cali,2024-03-11,103.2
742,dengue_cali,2024-03-18,89.8
743,dengue_cali,2024-03-25,1.0


---
## PARTE 4: EDA - ACF, PACF Y ESTACIONALIDAD

Siguiendo el flujo del Workshop S6, esta seccion revisa ACF y PACF antes de definir modelos. La ACF ayuda a leer persistencia y posibles ciclos; la PACF ayuda a identificar dependencias directas utiles para justificar ordenes AR y lags de machine learning.

Tambien se reportan los valores ACF en rezagos anuales y cuatrienales (`52`, `104`, `156`, `208`) para evaluar si hay soporte empirico para una estacionalidad anual o cuatrienal. En esta serie hay pocas repeticiones completas de un ciclo de 208 semanas, por lo que la evidencia cuatrienal debe interpretarse con cautela.


In [22]:
def graficar_acf_pacf(serie, n_lags=208, titulo="ACF y PACF"):
    """Grafica ACF y PACF al estilo del Workshop S6 y reporta lags significativos."""
    valores = np.asarray(serie).astype(float)
    valores = valores[~np.isnan(valores)]
    n_lags = min(n_lags, len(valores) // 2 - 1)

    acf_vals = acf(valores, nlags=n_lags, fft=True)
    pacf_vals = pacf(valores, nlags=n_lags, method="ols")
    conf_int = 1.96 / np.sqrt(len(valores))

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("ACF - autocorrelacion", "PACF - autocorrelacion parcial"),
    )

    for col_idx, (vals, color) in enumerate([(acf_vals, "#1f77b4"), (pacf_vals, "#2ca02c")], start=1):
        for lag, valor in enumerate(vals):
            bar_color = color if abs(valor) > conf_int else "lightgray"
            fig.add_trace(
                go.Bar(x=[lag], y=[valor], marker_color=bar_color, showlegend=False),
                row=1,
                col=col_idx,
            )
        fig.add_hline(y=conf_int, line_dash="dash", line_color="crimson", row=1, col=col_idx)
        fig.add_hline(y=-conf_int, line_dash="dash", line_color="crimson", row=1, col=col_idx)
        fig.add_hline(y=0, line_color="black", line_width=0.8, row=1, col=col_idx)

    fig.update_layout(title=titulo, template="plotly_white", height=430)
    fig.show()

    sig_acf = [lag for lag, valor in enumerate(acf_vals[1:], start=1) if abs(valor) > conf_int]
    sig_pacf = [lag for lag, valor in enumerate(pacf_vals[1:], start=1) if abs(valor) > conf_int]
    return acf_vals, pacf_vals, conf_int, sig_acf, sig_pacf


def tabla_acf_estacional(acf_vals, lags_interes=(52, 104, 156, 208)):
    """Resume autocorrelacion en rezagos anuales y cuatrienales."""
    filas = []
    for lag in lags_interes:
        if lag < len(acf_vals):
            filas.append({"lag": lag, "acf": acf_vals[lag], "interpretacion": "significativo" if abs(acf_vals[lag]) > 1.96 / np.sqrt(len(df)) else "no significativo"})
    return pd.DataFrame(filas)


print("Resumen descriptivo de la serie imputada:")
display(df["y"].describe().to_frame("casos"))

acf_vals, pacf_vals, conf_int, sig_acf, sig_pacf = graficar_acf_pacf(
    df.y,
    n_lags=ESTACIONALIDAD,
    titulo="ACF y PACF - Casos semanales de dengue en Cali",
)

print(f"IC aproximado 95%: +/- {conf_int:.3f}")
print(f"Primeros lags significativos ACF:  {sig_acf[:15]}")
print(f"Primeros lags significativos PACF: {sig_pacf[:15]}")
print("\nACF en rezagos de interes estacional:")
display(tabla_acf_estacional(acf_vals))

lags_sugeridos_ml = sorted(set([lag for lag in sig_pacf[:8] if lag <= 24] + [52]))
print(f"\nLags sugeridos por PACF + referencia anual: {lags_sugeridos_ml}")


Resumen descriptivo de la serie imputada:


,casos
count,744.000000
mean,83.666398
std,97.201164
min,1.000000
25%,17.000000
50%,51.000000
75%,107.000000
max,573.000000


IC aproximado 95%: +/- 0.072
Primeros lags significativos ACF:  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Primeros lags significativos PACF: [1, 2, 3, 4, 5, 8, 11, 13, 18, 20, 23, 24, 25, 26, 28]

ACF en rezagos de interes estacional:


,lag,acf,interpretacion
0,52,-0.092103,significativo
1,104,-0.169386,significativo
2,156,0.040873,no significativo
3,208,0.039015,no significativo



Lags sugeridos por PACF + referencia anual: [1, 2, 3, 4, 5, 8, 11, 13, 52]


In [23]:
# Serie temporal con coloreado de semanas epidemicas extremas
q1 = df.y.quantile(0.25)
q3 = df.y.quantile(0.75)
iqr = q3 - q1
umbral_brote = q3 + 1.5 * iqr

semanas_brote = df[df.y > umbral_brote]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df.ds, y=df.y, mode="lines",
    name="Casos semanales", line=dict(color="#1f77b4", width=1)
))
fig.add_trace(go.Scatter(
    x=semanas_brote.ds, y=semanas_brote.y, mode="markers",
    name="Semanas epidemicas extremas", marker=dict(color="#e74c3c", size=5, opacity=0.65)
))
fig.add_hline(
    y=umbral_brote, line_dash="dash", line_color="#e74c3c",
    annotation_text=f"Umbral IQR = {umbral_brote:.0f} casos", annotation_position="top left"
)
fig.update_layout(
    title="Serie Temporal: Casos Semanales de Dengue en Cali",
    xaxis_title="Semana epidemiologica", yaxis_title="Casos",
    hovermode="x unified", height=450, template="plotly_white"
)
fig.show()

print(f"Semanas epidemicas extremas detectadas: {len(semanas_brote)}")
print("Observacion: los puntos rojos se concentran en periodos de brote, no en ruido aislado.")
print("Esto motiva conservar los extremos y usar rezagos/ventanas moviles en los modelos ML.")


Semanas epidemicas extremas detectadas: 57
Observacion: los puntos rojos se concentran en periodos de brote, no en ruido aislado.
Esto motiva conservar los extremos y usar rezagos/ventanas moviles en los modelos ML.


In [24]:
# Descomposicion estacional: modelo aditivo vs multiplicativo
#
# Nota metodologica:
# La descomposicion multiplicativa requiere y_t > 0 en todos los puntos.
# En esta serie imputada los casos son positivos, pero dejamos la validacion explicita
# porque es una buena practica con datos epidemiologicos reales.

periodo = ESTACIONALIDAD  # dengue semanal: patron anual aproximado de 52 semanas

dec_add = seasonal_decompose(df.y, model="additive", period=periodo, extrapolate_trend="freq")
var_add = np.var(dec_add.resid.dropna())

try:
    if (df.y <= 0).any():
        raise ValueError(
            f"Serie contiene {(df.y <= 0).sum()} valores <= 0. "
            "Descomposicion multiplicativa no aplicable."
        )

    dec_mul = seasonal_decompose(df.y, model="multiplicative", period=periodo, extrapolate_trend="freq")
    var_mul = np.var(dec_mul.resid.dropna())
    mejor = "aditivo" if var_add < var_mul else "multiplicativo"
    dec = dec_add if mejor == "aditivo" else dec_mul

    print(f"Varianza residuos aditivo:        {var_add:.2f}")
    print(f"Varianza residuos multiplicativo: {var_mul:.2f}")
    print(f"Mejor modelo segun varianza residual: {mejor.upper()}")
except ValueError as e:
    print("Descomposicion multiplicativa descartada:")
    print(f"   {e}")
    print("   Usando modelo ADITIVO.")
    mejor = "aditivo"
    dec = dec_add

print(f"\nAnalisis cuantitativo - modelo {mejor.upper()}:")
print(f"  Fuerza de tendencia:    {np.std(dec.trend.dropna()) / np.std(dec.observed):.3f}")
print(f"  Fuerza estacional:      {np.std(dec.seasonal.dropna()) / np.std(dec.observed):.3f}")
print(f"  Amplitud estacional:    {dec.seasonal.max() - dec.seasonal.min():.1f} casos")
print(f"  Ruido residual (ratio): {np.std(dec.resid.dropna()) / np.std(dec.observed):.3f}")

fig = make_subplots(
    rows=4,
    cols=1,
    vertical_spacing=0.05,
    subplot_titles=("Serie original", "Tendencia", "Estacionalidad", "Residuos"),
)
componentes = [("observed", "#1f77b4"), ("trend", "#d62728"), ("seasonal", "#2ca02c"), ("resid", "#ff7f0e")]
for row, (comp, color) in enumerate(componentes, 1):
    fig.add_trace(
        go.Scatter(x=df.ds, y=getattr(dec, comp), mode="lines", showlegend=False, line=dict(color=color, width=1)),
        row=row,
        col=1,
    )

fig.update_layout(
    height=800,
    title=f"Descomposicion Estacional - Modelo {mejor.upper()} - periodo={periodo} semanas",
    template="plotly_white",
)
fig.show()

print("Observacion: si el componente estacional es debil o inestable, la estacionalidad fija debe tratarse como hipotesis predictiva, no como hecho epidemiologico definitivo.")


Varianza residuos aditivo:        7551.49
Varianza residuos multiplicativo: 0.90
Mejor modelo segun varianza residual: MULTIPLICATIVO

Analisis cuantitativo - modelo MULTIPLICATIVO:
  Fuerza de tendencia:    0.220
  Fuerza estacional:      0.006
  Amplitud estacional:    2.3 casos
  Ruido residual (ratio): 0.010


Observacion: si el componente estacional es debil o inestable, la estacionalidad fija debe tratarse como hipotesis predictiva, no como hecho epidemiologico definitivo.


---
## PARTE 5: ANALISIS DE ESTACIONARIEDAD

Aplicamos la prueba ADF antes y despues de transformaciones por diferencias, siguiendo la estructura del workshop.

**MLForecast** puede aplicar estas transformaciones automaticamente con `target_transforms`. En este proyecto no se modifica la serie original en esta seccion: las diferencias se usan como diagnostico para entender si la serie requiere transformar el objetivo antes de modelar.


In [ ]:
def adf_report(series, nombre):
    res = adfuller(series.dropna(), autolag="AIC")
    estacionaria = res[1] <= 0.05
    icono = "✅ ESTACIONARIA" if estacionaria else "❌ NO ESTACIONARIA"
    print(f"ADF — {nombre}:")
    print(f"  Estadístico: {res[0]:.4f}  |  p-valor: {res[1]:.4f}  →  {icono}")
    return estacionaria


adf_report(df.y, "Serie original imputada")

datos_diff52 = df.y.diff(52)
adf_report(datos_diff52, "Diff(52) — diferencia anual")

# Diferencia con la estacionalidad principal del notebook.
if ESTACIONALIDAD != 52:
    datos_diff_estacional = df.y.diff(ESTACIONALIDAD)
    adf_report(datos_diff_estacional, f"Diff({ESTACIONALIDAD}) — diferencia estacional candidata")

print(f"\n💡 La diferencia estacional puede aplicarse vía target_transforms=[Differences([{ESTACIONALIDAD}])]" )
print("   en el objeto MLForecast, sin necesidad de modificar el DataFrame original.")
print("   En este notebook se conserva la escala original para mantener interpretabilidad sanitaria.")


# Visualizar el efecto de distintas diferenciaciones usando MLForecast.preprocess()
def plot_differencias(datos_base, differences, freq="W-MON"):
    prep_list = [datos_base.assign(unique_id="original")]
    for d in differences:
        fcst_ = MLForecast(models=[], freq=freq, target_transforms=[Differences([d])])
        df_ = fcst_.preprocess(datos_base.copy(), static_features=[])
        df_["unique_id"] = f"diff({d})"
        prep_list.append(df_)
    prep = pd.concat(prep_list, ignore_index=True)

    fig = go.Figure()
    for uid in prep.unique_id.unique():
        sub = prep[prep.unique_id == uid]
        fig.add_trace(go.Scatter(x=sub.ds, y=sub.y, mode="lines", name=uid))
    fig.update_layout(
        title="Efecto de diferenciaciones sobre la serie",
        xaxis_title="Fecha",
        yaxis_title="Valor transformado",
        template="plotly_white",
        height=450,
    )
    fig.show()

plot_differencias(df[["unique_id", "ds", "y"]], differences=[1, 4, 28, 52], freq=FRECUENCIA)

# Complemento requerido para el proyecto: KPSS y Ljung-Box.
def test_estacionaridad(serie, nombre):
    """
    Aplica ADF + KPSS y entrega una conclusion conjunta.
    ADF p<0.05 + KPSS p>=0.05 indica estacionariedad compatible.
    """
    arr = np.asarray(serie).astype(float)
    arr = arr[~np.isnan(arr)]

    adf_stat, adf_p, _, _, _, _ = adfuller(arr, autolag="AIC")
    kpss_stat, kpss_p, _, _ = kpss(arr, regression="c", nlags="auto")

    adf_est = adf_p < 0.05
    kpss_est = kpss_p >= 0.05

    if adf_est and kpss_est:
        conclusion = "✅ ESTACIONARIA"
    elif not adf_est and not kpss_est:
        conclusion = "❌ NO ESTACIONARIA"
    else:
        conclusion = "❓ INCIERTA"

    print(f"\n{nombre}")
    print("-" * 70)
    print(f"ADF:  stat={adf_stat:8.4f} | p={adf_p:.4f}  -> {'estacionaria' if adf_est else 'no estacionaria'}")
    print(f"KPSS: stat={kpss_stat:8.4f} | p={kpss_p:.4f}  -> {'estacionaria' if kpss_est else 'no estacionaria'}")
    print(f"Conclusion conjunta: {conclusion}")
    return {"adf_p": adf_p, "kpss_p": kpss_p, "estacionaria": adf_est and kpss_est}

r_estacionariedad = test_estacionaridad(df.y, "Casos de dengue en Cali - Serie imputada")



ADF — Serie original imputada:
  Estadístico: -5.0818  |  p-valor: 0.0000  →  ✅ ESTACIONARIA
ADF — Diff(52) — diferencia anual:
  Estadístico: -4.5456  |  p-valor: 0.0002  →  ✅ ESTACIONARIA
ADF — Diff(208) — diferencia estacional candidata:
  Estadístico: -3.8879  |  p-valor: 0.0021  →  ✅ ESTACIONARIA

💡 La diferencia estacional puede aplicarse vía target_transforms=[Differences([208])]
   en el objeto MLForecast, sin necesidad de modificar el DataFrame original.
   En este notebook se conserva la escala original para mantener interpretabilidad sanitaria.



Casos de dengue en Cali - Serie imputada
----------------------------------------------------------------------
ADF:  stat= -5.0818 | p=0.0000  -> estacionaria
KPSS: stat=  0.1358 | p=0.1000  -> estacionaria
Conclusion conjunta: ✅ ESTACIONARIA

Ljung-Box - Serie imputada


,lag,q_stat,p_valor,hay_autocorrelacion
0,1,680.343848,5.625361e-150,True
1,6,3264.086657,0.000000e+00,True
2,12,4851.421001,0.000000e+00,True
3,18,5562.677688,0.000000e+00,True
4,24,5778.253770,0.000000e+00,True
5,52,5881.216601,0.000000e+00,True


In [39]:
def prueba_ljungbox(serie, lags_test=None):
    """Aplica Ljung-Box para varios rezagos."""
    if lags_test is None:
        lags_test = [1, 6, 12, 18, 24]

    valores = np.asarray(serie).astype(float)
    valores = valores[~np.isnan(valores)]

    resultados = []
    for lag in lags_test:
        result = acorr_ljungbox(valores, lags=[lag], return_df=True)
        q_stat = result["lb_stat"].iloc[0]
        p_val = result["lb_pvalue"].iloc[0]
        resultados.append({"lag": lag, "q_stat": q_stat, "p_valor": p_val, "hay_autocorrelacion": "✅" if p_val < 0.05 else "❌"})

    return pd.DataFrame(resultados)

print("\n" + "TEST DE LJUNG-BOX")
display(prueba_ljungbox(
    df.y.values,
    lags_test=[1, 6, 12, 18, 24, 52],
))


TEST DE LJUNG-BOX


,lag,q_stat,p_valor,hay_autocorrelacion
0,1,680.343848,5.625361e-150,✅
1,6,3264.086657,0.000000e+00,✅
2,12,4851.421001,0.000000e+00,✅
3,18,5562.677688,0.000000e+00,✅
4,24,5778.253770,0.000000e+00,✅
5,52,5881.216601,0.000000e+00,✅


---
## PARTE 6: BASELINES DE FASE 1

Los baselines se reproducen con las mismas funciones de fase 1. En fase 2 se usan como referencia cuantitativa para medir mejora porcentual, no como modelos candidatos principales.

In [26]:
def crear_statsforecast_baseline(frecuencia="W-MON", estacionalidad=ESTACIONALIDAD):
    """Crea el objeto StatsForecast con los modelos baseline del taller."""
    return StatsForecast(
        models=[
            Naive(),
            SeasonalNaive(season_length=estacionalidad),
            WindowAverage(window_size=3),
            RandomWalkWithDrift()
        ],
        freq=frecuencia
    )

def entrenar_y_predecir_baselines(train, test, frecuencia="W-MON", estacionalidad=ESTACIONALIDAD):
    """Entrena los baselines y predice el horizonte del test."""
    horizonte = len(test)
    sf = crear_statsforecast_baseline(frecuencia=frecuencia, estacionalidad=estacionalidad)
    sf.fit(train)
    preds = sf.predict(h=horizonte)

    print("Pronósticos generados:")
    print(f"  Modelos: {[c for c in preds.columns if c not in ['unique_id', 'ds']]}")
    print(f"  Horizonte: {horizonte} semanas")
    display(preds.head())
    return sf, preds

def unir_predicciones_con_test(test, preds):
    """Une los valores reales del test con las predicciones."""
    return test.merge(preds, on=["unique_id", "ds"])

def graficar_pronosticos_baseline(serie, test, test_preds, fecha_corte, modelos_col):
    """Grafica histórico, test y pronósticos de cada baseline."""
    colores = ["#FF5722", "#4CAF50", "#9C27B0", "#FF9800"]
    fecha_corte = pd.to_datetime(fecha_corte)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.ds, y=serie.y, mode="lines", name="Histórico",
        line=dict(color="lightgray", width=1)
    ))
    fig.add_trace(go.Scatter(
        x=test.ds, y=test.y, mode="lines+markers", name="Real (test)",
        line=dict(color="black", width=1), marker=dict(size=2)
    ))

    for modelo, color in zip(modelos_col, colores):
        fig.add_trace(go.Scatter(
            x=test_preds.ds, y=test_preds[modelo], mode="lines+markers", name=modelo,
            line=dict(color=color, width=1, dash="dot"), marker=dict(size=2)
        ))

    fig.add_shape(
        type="line", xref="x", yref="paper",
        x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
        line=dict(dash="dash", color="gray", width=1)
    )
    fig.add_annotation(
        x=fecha_corte, y=1.02, yref="paper",
        text="Inicio test", showarrow=False,
        font=dict(size=10, color="gray"), xanchor="left"
    )
    fig.update_layout(
        title="Comparación de Baselines — Casos Dengue",
        xaxis_title="Fecha", yaxis_title="Casos",
        height=480, template="plotly_white",
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.show()


def calcular_metricas_pronostico(datos, modelos, conjunto="test"):
    """Calcula MAE, RMSE y MAPE con utilsforecast, igual que en el ecosistema Nixtla."""
    evaluacion = evaluate(
        df=datos,
        metrics=[mae, rmse, mape],
        models=modelos,
        target_col="y",
    )

    columnas_metricas = ["metric"] + [modelo for modelo in modelos if modelo in evaluacion.columns]
    tabla = (
        evaluacion[columnas_metricas]
        .melt(id_vars="metric", var_name="modelo", value_name="valor")
        .pivot_table(index="modelo", columns="metric", values="valor", aggfunc="mean")
        .reset_index()
    )
    tabla.columns.name = None
    tabla["mape"] = tabla["mape"] * 100
    tabla.insert(1, "conjunto", conjunto)
    return tabla.sort_values("rmse").reset_index(drop=True)


sf_baseline, preds_baseline = entrenar_y_predecir_baselines(
    train=train,
    test=test,
    frecuencia=FRECUENCIA,
    estacionalidad=ESTACIONALIDAD,
)

test_baseline = unir_predicciones_con_test(test, preds_baseline)
modelos_baseline = [c for c in preds_baseline.columns if c not in ["unique_id", "ds"]]

graficar_pronosticos_baseline(df, test, test_baseline, FECHA_CORTE, modelos_baseline)
metricas_baseline = calcular_metricas_pronostico(test_baseline, modelos_baseline)
display(metricas_baseline)

Pronósticos generados:
  Modelos: ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
  Horizonte: 221 semanas


,unique_id,ds,Naive,SeasonalNaive,WindowAverage,RWD
0,dengue_cali,2020-01-06,162.0,216.0,141.666667,162.287356
1,dengue_cali,2020-01-13,162.0,281.0,141.666667,162.574713
2,dengue_cali,2020-01-20,162.0,272.0,141.666667,162.862069
3,dengue_cali,2020-01-27,162.0,310.0,141.666667,163.149425
4,dengue_cali,2020-02-03,162.0,365.0,141.666667,163.436782


,modelo,conjunto,mae,mape,rmse
0,WindowAverage,test,99.925490,299.540207,115.617042
1,SeasonalNaive,test,81.828959,369.357753,121.823961
2,Naive,test,111.987330,351.029934,123.140951
3,RWD,test,130.194695,464.707735,140.111914


---
## PARTE 7: ARIMA/SARIMA Y SUAVIZACION EXPONENCIAL

Se implementan cuatro modelos estadisticos avanzados:

- `AutoARIMA`: seleccion automatica con estacionalidad.
- `SARIMA_manual`: parametrizacion manual `(2,0,2)(1,0,1)[ESTACIONALIDAD]`, consistente con persistencia de corto plazo y componente anual potencial.
- `AutoETS`: suavizacion exponencial optimizada.
- `Theta`: modelo Theta optimizado con estacionalidad.

In [27]:
def entrenar_y_predecir_statsforecast(train, test, modelos, frecuencia=FRECUENCIA):
    """Entrena modelos StatsForecast y mide tiempo total por familia."""
    inicio = time.perf_counter()
    sf = StatsForecast(models=modelos, freq=frecuencia, n_jobs=-1)
    sf.fit(train)
    preds = sf.predict(h=len(test))
    tiempo = time.perf_counter() - inicio
    return sf, preds, tiempo

#cambiar a estcionalida_anual si es necesario
modelos_stats = [
    AutoARIMA(season_length=ESTACIONALIDAD, alias=f'AutoARIMA_{ESTACIONALIDAD}'),
    ARIMA(order=(2, 0, 2), seasonal_order=(1, 0, 1), season_length=ESTACIONALIDAD, alias=f'SARIMA_manual'),
    AutoETS(season_length=ESTACIONALIDAD, model="ZZZ", alias=f'AutoETS_{ESTACIONALIDAD}'),
    DynamicOptimizedTheta(season_length=ESTACIONALIDAD, alias=f'Theta_{ESTACIONALIDAD}'),
]

sf_stats, preds_stats, tiempo_stats = entrenar_y_predecir_statsforecast(train, test, modelos_stats)
test_stats = unir_predicciones_con_test(test, preds_stats)
modelos_stats_col = [c for c in preds_stats.columns if c not in ["unique_id", "ds"]]

test_stats[modelos_stats_col] = test_stats[modelos_stats_col].clip(lower=0)
metricas_stats = calcular_metricas_pronostico(test_stats, modelos_stats_col)
metricas_stats["tiempo_segundos"] = tiempo_stats

display(metricas_stats)

,modelo,conjunto,mae,mape,rmse,tiempo_segundos
0,SARIMA_manual,test,61.937628,117.060029,109.005144,22.719304
1,Theta_208,test,111.825577,350.364690,123.026029,22.719304
2,AutoARIMA_208,test,118.391996,377.178595,127.950773,22.719304
3,AutoETS_208,test,162.264421,653.503775,179.448633,22.719304


---
## PARTE 8: FEATURE ENGINEERING CON `MLForecast.preprocess()`

Aqui usamos `MLForecast.preprocess()` para inspeccionar exactamente que variables recibe el modelo.

La adaptacion para dengue usa tres grupos de features:

- **Lags:** seleccionados con PACF y una referencia anual (`52`).
- **Ventanas deslizantes:** media, desviacion, minimo y maximo para capturar nivel local y volatilidad.
- **Calendario/Fourier:** terminos ciclicos anuales y una aproximacion cuatrienal suave.

> Esto es lo que el modelo realmente ve cuando entrenamos.


In [ ]:
# Definimos la configuracion de features — usada por TODOS los modelos ML
FREQ = FRECUENCIA
LAGS = lags_sugeridos_ml              # justificados por PACF + referencia anual
VENTANAS_ROLLING = [4, 8, 12, 24]
TRANSFORMACIONES_ROLLING = [RollingMean, RollingStd, RollingMin, RollingMax]

LAG_TR = {
    1: [
        transformacion(window_size=ventana)
        for ventana in VENTANAS_ROLLING
        for transformacion in TRANSFORMACIONES_ROLLING
    ]
}

# Terminos Fourier: reemplazan codificaciones ordinales rigidas del ciclo anual.
# Para datos semanales, season_length=52 representa el ciclo epidemiologico anual.
# k=2 genera 4 columnas: sin1_52, sin2_52, cos1_52, cos2_52.
df_fourier, _ = fourier(
    df,
    freq=FREQ,
    season_length=ESTACIONALIDAD,
    k=2,
    h=1,
)
FOURIER_COLS = [c for c in df_fourier.columns if c not in df.columns]
print(f"Terminos Fourier generados: {FOURIER_COLS}")
print(f"df_fourier shape: {df_fourier.shape} (historico con Fourier anual)")


def agregar_variables_calendario(serie):
    """Agrega variables dinamicas conocidas para train y test."""
    datos = serie.copy().sort_values("ds").reset_index(drop=True)
    iso = datos.ds.dt.isocalendar()
    datos["semana_iso"] = iso.week.astype(int)

    # Ciclo cuatrienal aproximado: posicion temporal dentro de bloques de 208 semanas.
    t = np.arange(len(datos))
    datos["ciclo_sin_208"] = np.sin(2 * np.pi * t / ESTACIONALIDAD)
    datos["ciclo_cos_208"] = np.cos(2 * np.pi * t / ESTACIONALIDAD)
    return datos


df_ml = agregar_variables_calendario(df_fourier)
CALENDAR_COLS = ["semana_iso", "ciclo_sin_208", "ciclo_cos_208"]
DATE_F = ["month", "quarter", "year"]


def crear_mlforecast(modelos):
    return MLForecast(
        models=modelos,
        freq=FREQ,
        lags=LAGS,
        lag_transforms=LAG_TR,
        date_features=DATE_F,
    )


# Instanciamos MLForecast sin modelos solo para inspeccionar los features.
# IMPORTANTE: static_features=[] porque Fourier y calendario cambian en el tiempo.
fcst_explorer = crear_mlforecast(modelos=[])
features_df = fcst_explorer.preprocess(
    df_ml[["unique_id", "ds", "y"] + FOURIER_COLS + CALENDAR_COLS],
    static_features=[],
)

print(f"\nDimensiones del dataset de features: {features_df.shape}")
print("Columnas generadas:")
feature_cols = [c for c in features_df.columns if c not in ["unique_id", "ds", "y"]]
for c in feature_cols:
    print(f"  • {c}")
display(features_df.tail(5))

train_ml = df_ml[df_ml.ds < pd.to_datetime(FECHA_CORTE)].copy()
test_ml = df_ml[df_ml.ds >= pd.to_datetime(FECHA_CORTE)].copy()
X_test_ml = test_ml.drop(columns="y")


Terminos Fourier generados: ['sin1_208', 'sin2_208', 'cos1_208', 'cos2_208']
df_fourier shape: (744, 7) (historico con Fourier anual)

Dimensiones del dataset de features: (692, 38)
Columnas generadas:
  • sin1_208
  • sin2_208
  • cos1_208
  • cos2_208
  • semana_iso
  • ciclo_sin_208
  • ciclo_cos_208
  • lag1
  • lag2
  • lag3
  • lag4
  • lag5
  • lag8
  • lag11
  • lag13
  • lag52
  • rolling_mean_lag1_window_size4
  • rolling_std_lag1_window_size4
  • rolling_min_lag1_window_size4
  • rolling_max_lag1_window_size4
  • rolling_mean_lag1_window_size8
  • rolling_std_lag1_window_size8
  • rolling_min_lag1_window_size8
  • rolling_max_lag1_window_size8
  • rolling_mean_lag1_window_size12
  • rolling_std_lag1_window_size12
  • rolling_min_lag1_window_size12
  • rolling_max_lag1_window_size12
  • rolling_mean_lag1_window_size24
  • rolling_std_lag1_window_size24
  • rolling_min_lag1_window_size24
  • rolling_max_lag1_window_size24
  • month
  • quarter
  • year


,unique_id,ds,y,sin1_208,sin2_208,cos1_208,cos2_208,semana_iso,ciclo_sin_208,ciclo_cos_208,...,rolling_std_lag1_window_size12,rolling_min_lag1_window_size12,rolling_max_lag1_window_size12,rolling_mean_lag1_window_size24,rolling_std_lag1_window_size24,rolling_min_lag1_window_size24,rolling_max_lag1_window_size24,month,quarter,year
739,dengue_cali,2024-02-26,121.8,-0.354604,0.663122,-0.935016,0.748511,9,-0.326203,-0.945300,...,220.606653,3.0,537.0,259.250000,175.089108,3.0,537.0,2,1,2024
740,dengue_cali,2024-03-04,5.0,-0.382682,0.707105,-0.923880,0.707109,10,-0.354605,-0.935016,...,207.720132,3.0,537.0,249.366667,175.906424,3.0,537.0,3,1,2024
741,dengue_cali,2024-03-11,103.2,-0.410412,0.748510,-0.911900,0.663123,11,-0.382683,-0.923880,...,190.429985,3.0,537.0,233.658333,180.324832,3.0,537.0,3,1,2024
742,dengue_cali,2024-03-18,89.8,-0.437766,0.787182,-0.899089,0.616721,12,-0.410413,-0.911900,...,155.947951,3.0,537.0,223.041667,180.186550,3.0,537.0,3,1,2024
743,dengue_cali,2024-03-25,1.0,-0.464723,0.822983,-0.885456,0.568065,13,-0.437768,-0.899088,...,67.167802,3.0,213.0,213.033333,180.657578,3.0,537.0,3,1,2024


In [43]:
# Mapa de correlaciones entre features y target
corr = (
    features_df[feature_cols + ["y"]]
    .corr(numeric_only=True)["y"]
    .drop("y")
    .sort_values(key=abs, ascending=False)
)

fig = go.Figure(go.Bar(
    x=corr.values,
    y=corr.index,
    orientation="h",
    marker_color=["#e74c3c" if v < 0 else "#2ecc71" for v in corr.values],
))
fig.update_layout(
    title="Correlacion de Features con el Target (casos de dengue)",
    xaxis_title="Correlacion de Pearson",
    yaxis_title="Feature",
    height=520,
    template="plotly_white",
)
fig.show()
print("Los lags y ventanas con mayor correlacion suelen ser los mas informativos para modelos de boosting.")


Los lags y ventanas con mayor correlacion suelen ser los mas informativos para modelos de boosting.


---
## PARTE 9: MODELOS ML Y OPTIMIZACION TEMPORAL

Se entrenan cuatro modelos obligatorios: dos lineales regularizados y dos ensembles. La optimizacion se hace con validacion temporal corta, evaluando pocos candidatos para mantener el notebook reproducible y legible.

In [30]:
def evaluar_candidato_ml(nombre, modelo, train_ml, h=26, n_windows=3, step_size=26):
    """Evalua un candidato MLForecast por RMSE promedio en CV temporal."""
    fcst = crear_mlforecast({nombre: modelo})
    cv = fcst.cross_validation(
        df=train_ml,
        h=h,
        n_windows=n_windows,
        step_size=step_size,
        refit=True,
        static_features=[],
    )
    metricas = calcular_metricas_pronostico(cv, [nombre], conjunto="cv")
    return metricas.loc[0, "rmse"]


def optimizar_modelos_ml(train_ml):
    """Selecciona hiperparametros simples mediante validacion temporal."""
    candidatos = {
        "ridge": [
            ("alpha=0.1", make_pipeline(StandardScaler(), Ridge(alpha=0.1))),
            ("alpha=1.0", make_pipeline(StandardScaler(), Ridge(alpha=1.0))),
            ("alpha=10.0", make_pipeline(StandardScaler(), Ridge(alpha=10.0))),
        ],
        "lasso": [
            ("alpha=0.01", make_pipeline(StandardScaler(), Lasso(alpha=0.01, max_iter=10000, random_state=SEED))),
            ("alpha=0.05", make_pipeline(StandardScaler(), Lasso(alpha=0.05, max_iter=10000, random_state=SEED))),
            ("alpha=0.10", make_pipeline(StandardScaler(), Lasso(alpha=0.10, max_iter=10000, random_state=SEED))),
        ],
        "random_forest": [
            ("depth=5_leaf=3", RandomForestRegressor(n_estimators=150, max_depth=5, min_samples_leaf=3, random_state=SEED, n_jobs=-1)),
            ("depth=8_leaf=3", RandomForestRegressor(n_estimators=150, max_depth=8, min_samples_leaf=3, random_state=SEED, n_jobs=-1)),
        ],
        "lightgbm": [
            ("n=250_lr=0.03_leaves=15", LGBMRegressor(n_estimators=250, learning_rate=0.03, num_leaves=15, min_child_samples=15, random_state=SEED, n_jobs=-1, verbose=-1)),
            ("n=500_lr=0.03_leaves=31", LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=15, random_state=SEED, n_jobs=-1, verbose=-1)),
        ],
    }

    filas = []
    mejores = {}
    for nombre, opciones in candidatos.items():
        for etiqueta, modelo in opciones:
            rmse_cv = evaluar_candidato_ml(nombre, modelo, train_ml)
            filas.append({"modelo": nombre, "parametros": etiqueta, "rmse_cv": rmse_cv})
        mejor = min([f for f in filas if f["modelo"] == nombre], key=lambda x: x["rmse_cv"])
        mejores[nombre] = dict(opciones)[mejor["parametros"]]

    return mejores, pd.DataFrame(filas).sort_values(["modelo", "rmse_cv"])


modelos_ml, resultados_tuning_ml = optimizar_modelos_ml(train_ml)
print("Mejores hiperparametros por modelo:")
display(resultados_tuning_ml.groupby("modelo").head(1))

Mejores hiperparametros por modelo:


,modelo,parametros,rmse_cv
5,lasso,alpha=0.10,28.022783
8,lightgbm,n=250_lr=0.03_leaves=15,23.698748
7,random_forest,depth=8_leaf=3,23.189601
2,ridge,alpha=10.0,29.677510


In [31]:
def entrenar_y_predecir_ml(train_ml, test_ml, modelos_ml):
    """Entrena MLForecast y genera predicciones sobre test."""
    inicio = time.perf_counter()
    fcst = crear_mlforecast(modelos_ml)
    fcst.fit(train_ml, static_features=[])
    preds = fcst.predict(h=len(test_ml), X_df=X_test_ml)
    tiempo = time.perf_counter() - inicio
    return fcst, preds, tiempo


fcst_ml, preds_ml, tiempo_ml = entrenar_y_predecir_ml(train_ml, test_ml, modelos_ml)
test_ml_pred = unir_predicciones_con_test(test, preds_ml)
modelos_ml_col = [c for c in preds_ml.columns if c not in ["unique_id", "ds"]]
test_ml_pred[modelos_ml_col] = test_ml_pred[modelos_ml_col].clip(lower=0)

metricas_ml = calcular_metricas_pronostico(test_ml_pred, modelos_ml_col)
metricas_ml["tiempo_segundos"] = tiempo_ml

display(metricas_ml)

,modelo,conjunto,mae,mape,rmse,tiempo_segundos
0,lightgbm,test,90.115758,373.248942,101.617233,7.561116
1,random_forest,test,115.317569,387.912911,124.986617,7.561116
2,lasso,test,88.877820,82.720795,134.086259,7.561116
3,ridge,test,91.290966,84.753711,136.269805,7.561116


---
## PARTE 10: CROSS-VALIDATION Y BENCHMARK INTEGRAL

El benchmark combina resultados en test con una validacion rolling-origin. Esto permite revisar precision fuera de muestra y estabilidad temporal antes de seleccionar el modelo final.

In [32]:
def ejecutar_validacion_cruzada(sf, serie, h=52, n_windows=3, step_size=52):
    """Ejecuta cross-validation temporal con StatsForecast."""
    cv_results = sf.cross_validation(
        df=serie,
        h=h,
        n_windows=n_windows,
        step_size=step_size
    )

    print(f"Resultados CV: {cv_results.shape[0]} filas × {cv_results.shape[1]} columnas")
    print(f"Folds (cutoffs): {cv_results.cutoff.unique().tolist()}")
    display(cv_results.head(6))
    return cv_results


def metricas_cv_por_modelo(cv, modelos):
    filas = []
    for cutoff, sub in cv.groupby("cutoff"):
        m = calcular_metricas_pronostico(sub, modelos, conjunto="cv")
        m["cutoff"] = cutoff
        filas.append(m)
    detalle = pd.concat(filas, ignore_index=True)
    resumen = detalle.groupby("modelo", as_index=False)[["mae", "rmse", "mape"]].mean()
    return detalle, resumen.sort_values("rmse")


cv_stats = ejecutar_validacion_cruzada(sf_stats, train, h=52, n_windows=3, step_size=52)
_, metricas_cv_stats = metricas_cv_por_modelo(cv_stats, modelos_stats_col)

cv_ml = fcst_ml.cross_validation(
    df=train_ml,
    h=52,
    n_windows=3,
    step_size=52,
    refit=True,
    static_features=[],
)
_, metricas_cv_ml = metricas_cv_por_modelo(cv_ml, modelos_ml_col)

metricas_cv = pd.concat([metricas_cv_stats, metricas_cv_ml], ignore_index=True).sort_values("rmse")
display(metricas_cv)

Resultados CV: 156 filas × 8 columnas
Folds (cutoffs): [Timestamp('2017-01-02 00:00:00'), Timestamp('2018-01-01 00:00:00'), Timestamp('2018-12-31 00:00:00')]


,unique_id,ds,cutoff,y,AutoARIMA_208,SARIMA_manual,AutoETS_208,Theta_208
0,dengue_cali,2017-01-09,2017-01-02,19.0,38.947430,39.230174,35.908195,42.858715
1,dengue_cali,2017-01-16,2017-01-02,40.0,45.578969,41.246100,35.918227,42.858715
2,dengue_cali,2017-01-23,2017-01-02,35.0,51.986163,42.083492,35.926253,42.858715
3,dengue_cali,2017-01-30,2017-01-02,44.0,54.488073,43.842649,35.932673,42.858715
4,dengue_cali,2017-02-06,2017-01-02,25.0,64.179181,45.476073,35.937810,42.858715
5,dengue_cali,2017-02-13,2017-01-02,21.0,66.303695,46.426803,35.941919,42.858715


,modelo,mae,rmse,mape
0,AutoETS_208,19.667565,25.659489,327.523320
1,Theta_208,22.419503,28.503742,198.761684
2,AutoARIMA_208,35.911770,42.504404,403.740610
4,random_forest,37.913777,44.753279,504.620953
5,lightgbm,38.596077,45.116094,486.975688
3,SARIMA_manual,61.240465,70.763833,1356.959118
6,ridge,70.466424,78.767202,861.092515
7,lasso,71.894331,80.986385,881.216014


In [33]:
def agregar_mejoras_vs_baseline(metricas, baseline):
    """Agrega mejoras porcentuales frente a un baseline especifico."""
    metricas = metricas.copy()
    base = metricas.loc[metricas.modelo == baseline].iloc[0]
    for metrica in ["mae", "rmse", "mape"]:
        metricas[f"mejora_{metrica}_vs_{baseline}"] = (base[metrica] - metricas[metrica]) / base[metrica] * 100
    return metricas


test_resultados = test.copy()
for preds in [preds_baseline, preds_stats, preds_ml]:
    test_resultados = test_resultados.merge(preds, on=["unique_id", "ds"], how="left")

modelos_todos = [c for c in test_resultados.columns if c not in ["unique_id", "ds", "y"]]
test_resultados[modelos_todos] = test_resultados[modelos_todos].clip(lower=0)

metricas_test = calcular_metricas_pronostico(test_resultados, modelos_todos)
metricas_test = metricas_test.merge(
    pd.DataFrame({
        "modelo": modelos_baseline + modelos_stats_col + modelos_ml_col,
        "familia": ["baseline"] * len(modelos_baseline) + ["statsforecast"] * len(modelos_stats_col) + ["mlforecast"] * len(modelos_ml_col),
        "tiempo_segundos": [np.nan] * len(modelos_baseline) + [tiempo_stats] * len(modelos_stats_col) + [tiempo_ml] * len(modelos_ml_col),
    }),
    on="modelo",
    how="left",
)
metricas_test = agregar_mejoras_vs_baseline(metricas_test, "SeasonalNaive")
metricas_test = agregar_mejoras_vs_baseline(metricas_test, "WindowAverage")

display(metricas_test.sort_values("rmse"))

,modelo,conjunto,mae,mape,rmse,familia,tiempo_segundos,mejora_mae_vs_SeasonalNaive,mejora_rmse_vs_SeasonalNaive,mejora_mape_vs_SeasonalNaive,mejora_mae_vs_WindowAverage,mejora_rmse_vs_WindowAverage,mejora_mape_vs_WindowAverage
0,lightgbm,test,90.115758,373.248942,101.617233,mlforecast,7.561116,-10.126975,16.586825,-1.053502,9.817047,12.108776,-24.607293
1,SARIMA_manual,test,61.937628,117.060029,109.005144,statsforecast,22.719304,24.308425,10.522410,68.307142,38.016188,5.718792,60.920095
2,WindowAverage,test,99.925490,299.540207,115.617042,baseline,NaN,-22.115069,5.094990,18.902418,0.000000,0.000000,0.000000
3,SeasonalNaive,test,81.828959,369.357753,121.823961,baseline,NaN,0.000000,0.000000,0.000000,18.110025,-5.368515,-23.308239
4,Theta_208,test,111.825577,350.364690,123.026029,statsforecast,22.719304,-36.657704,-0.986726,5.142186,-11.908961,-6.408213,-16.967499
5,Naive,test,111.987330,351.029934,123.140951,baseline,NaN,-36.855377,-1.081060,4.962078,-12.070834,-6.507612,-17.189588
6,random_forest,test,115.317569,387.912911,124.986617,mlforecast,7.561116,-40.925132,-2.596087,-5.023628,-15.403556,-8.103974,-29.502785
7,AutoARIMA_208,test,118.391996,377.178595,127.950773,statsforecast,22.719304,-44.682270,-5.029234,-2.117417,-18.480275,-10.667745,-25.919188
8,lasso,test,88.877820,82.720795,134.086259,mlforecast,7.561116,-8.614139,-10.065588,77.604154,11.055908,-15.974476,72.384076
9,ridge,test,91.290966,84.753711,136.269805,mlforecast,7.561116,-11.563152,-11.857966,77.053762,8.640963,-17.863078,71.705397


---
## PARTE 11: PRONOSTICOS, RESIDUOS Y SELECCION FINAL

La seleccion final considera RMSE, MAE, MAPE, mejora frente a baselines, residuos y estabilidad CV/test. En dengue, RMSE es relevante porque penaliza mas los errores grandes durante brotes.

In [34]:
def graficar_pronosticos_modelos(test_resultados, modelos, titulo):
    print(modelos)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=test_resultados.ds, y=test_resultados.y, name="Real", line=dict(color="black", width=3)))
    colores = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
    for modelo, color in zip(modelos, colores):
        fig.add_trace(go.Scatter(x=test_resultados.ds, y=test_resultados[modelo], name=modelo, line=dict(color=color, width=1.5)))
    fig.update_layout(title=titulo, xaxis_title="Semana", yaxis_title="Casos", template="plotly_white", height=520, hovermode="x unified")
    fig.show()
    


def graficar_metricas_benchmark(metricas):
    orden = metricas.sort_values("rmse")
    fig = make_subplots(rows=1, cols=3, subplot_titles=["MAE", "RMSE", "MAPE"])
    for i, metrica in enumerate(["mae", "rmse", "mape"], start=1):
        fig.add_trace(go.Bar(x=orden.modelo, y=orden[metrica]), row=1, col=i)
    fig.update_layout(title="Benchmark integral de modelos", template="plotly_white", height=420, showlegend=False)
    fig.update_xaxes(tickangle=45)
    fig.show()


modelos_grafica = ["SeasonalNaive", "WindowAverage"] + MODELOS_FASE2_STATS + MODELOS_FASE2_ML
graficar_pronosticos_modelos(test_resultados, modelos_grafica, "Pronosticos fase 2 vs casos reales")
graficar_metricas_benchmark(metricas_test)

['SeasonalNaive', 'WindowAverage', 'AutoARIMA_208', 'SARIMA_manual', 'AutoETS_208', 'Theta_208', 'ridge', 'lasso', 'random_forest', 'lightgbm']


In [35]:
def diagnosticar_residuos_modelos(datos, modelos):
    """Diagnostico estadistico compacto de residuos por modelo."""
    filas = []
    for modelo in modelos:
        sub = datos[["ds", "y", modelo]].dropna()
        residuos = sub["y"] - sub[modelo]
        lag_lb = min(24, max(1, len(residuos) // 5))

        jb_p = jarque_bera(residuos)[1]
        lb_p = acorr_ljungbox(residuos, lags=[lag_lb], return_df=True)["lb_pvalue"].iloc[0]
        adf_p = adfuller(residuos, autolag="AIC")[1]
        kpss_p = kpss(residuos, regression="c", nlags="auto")[1]
        dw = durbin_watson(residuos)
        exog_bp = pd.DataFrame({"prediccion": sub[modelo].values, "tendencia_t": np.arange(len(sub))})
        bp_p = het_breuschpagan(residuos.values, add_constant(exog_bp, has_constant="add"))[3]

        filas.append({
            "modelo": modelo,
            "media_residuo": residuos.mean(),
            "std_residuo": residuos.std(),
            "jarque_bera_p": jb_p,
            "ljung_box_p": lb_p,
            "adf_p": adf_p,
            "kpss_p": kpss_p,
            "durbin_watson": dw,
            "breusch_pagan_p": bp_p,
        })
    return pd.DataFrame(filas)


def graficar_residuos_modelo(datos, modelo):
    sub = datos[["ds", "y", modelo]].dropna()
    residuos = sub["y"] - sub[modelo]
    acf_vals = acf(residuos, nlags=40)

    fig = make_subplots(rows=2, cols=2, subplot_titles=["Residuos vs tiempo", "Histograma", "Q-Q plot", "ACF residuos"])
    fig.add_trace(go.Scatter(x=sub.ds, y=residuos, mode="lines", name="Residuo"), row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
    fig.add_trace(go.Histogram(x=residuos, nbinsx=30), row=1, col=2)
    osm, osr = stats.probplot(residuos, dist="norm", fit=False)
    fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers"), row=2, col=1)
    fig.add_trace(go.Bar(x=list(range(len(acf_vals))), y=acf_vals), row=2, col=2)
    fig.update_layout(title=f"Diagnostico de residuos - {modelo}", template="plotly_white", height=720, showlegend=False)
    fig.show()


diagnostico_residuos = diagnosticar_residuos_modelos(test_resultados, modelos_todos)
display(diagnostico_residuos.sort_values("modelo"))

,modelo,media_residuo,std_residuo,jarque_bera_p,ljung_box_p,adf_p,kpss_p,durbin_watson,breusch_pagan_p
4,AutoARIMA_208,-64.534392,110.734808,3.378664e-34,5.282677e-248,0.004545,0.100000,0.108941,7.579088e-07
6,AutoETS_208,-134.466743,119.099500,2.592488e-14,1.941176e-233,0.038881,0.022432,0.055620,4.438954e-13
0,Naive,-54.340271,110.753476,1.824945e-34,2.882011e-247,0.004876,0.100000,0.117977,3.268628e-06
3,RWD,-86.236823,110.679667,5.002397e-26,5.719434e-227,0.007773,0.100000,0.091156,3.881784e-09
5,SARIMA_manual,31.498999,104.591750,1.535754e-51,2.532475e-250,0.002732,0.055078,0.149733,9.512913e-17
1,SeasonalNaive,46.315837,112.931968,1.181488e-21,9.795050e-143,0.003261,0.100000,0.167637,1.438692e-36
7,Theta_208,-54.079342,110.753476,1.824945e-34,2.882011e-247,0.004876,0.100000,0.118198,3.361174e-06
2,WindowAverage,-34.006938,110.753476,1.824945e-34,2.882011e-247,0.004876,0.100000,0.133832,2.779524e-05
9,lasso,88.713394,100.772063,4.866178e-59,6.988928e-239,0.005340,0.063194,0.099564,1.625362e-13
11,lightgbm,-53.413555,86.643077,1.710124e-34,1.722414e-159,0.003257,0.100000,0.177204,1.242809e-21


In [36]:
comparacion_cv_test = metricas_cv.merge(
    metricas_test[["modelo", "mae", "rmse", "mape"]].rename(columns={"mae": "mae_test", "rmse": "rmse_test", "mape": "mape_test"}),
    on="modelo",
    how="inner",
)
comparacion_cv_test = comparacion_cv_test.rename(columns={"mae": "mae_cv", "rmse": "rmse_cv", "mape": "mape_cv"})
comparacion_cv_test["ratio_rmse_test_cv"] = comparacion_cv_test["rmse_test"] / comparacion_cv_test["rmse_cv"]

display(comparacion_cv_test.sort_values("ratio_rmse_test_cv", ascending=False))

ranking_final = metricas_test.sort_values(["rmse", "mae"]).reset_index(drop=True)
mejor_modelo = ranking_final.loc[0, "modelo"]

print(f"Mejor modelo por RMSE: {mejor_modelo}")
display(ranking_final.head(8))
graficar_residuos_modelo(test_resultados, mejor_modelo)

,modelo,mae_cv,rmse_cv,mape_cv,mae_test,rmse_test,mape_test,ratio_rmse_test_cv
0,AutoETS_208,19.667565,25.659489,327.523320,162.264421,179.448633,653.503775,6.993461
1,Theta_208,22.419503,28.503742,198.761684,111.825577,123.026029,350.364690,4.316136
2,AutoARIMA_208,35.911770,42.504404,403.740610,118.391996,127.950773,377.178595,3.010294
3,random_forest,37.913777,44.753279,504.620953,115.317569,124.986617,387.912911,2.792792
4,lightgbm,38.596077,45.116094,486.975688,90.115758,101.617233,373.248942,2.252350
6,ridge,70.466424,78.767202,861.092515,91.290966,136.269805,84.753711,1.730032
7,lasso,71.894331,80.986385,881.216014,88.877820,134.086259,82.720795,1.655664
5,SARIMA_manual,61.240465,70.763833,1356.959118,61.937628,109.005144,117.060029,1.540408


Mejor modelo por RMSE: lightgbm


,modelo,conjunto,mae,mape,rmse,familia,tiempo_segundos,mejora_mae_vs_SeasonalNaive,mejora_rmse_vs_SeasonalNaive,mejora_mape_vs_SeasonalNaive,mejora_mae_vs_WindowAverage,mejora_rmse_vs_WindowAverage,mejora_mape_vs_WindowAverage
0,lightgbm,test,90.115758,373.248942,101.617233,mlforecast,7.561116,-10.126975,16.586825,-1.053502,9.817047,12.108776,-24.607293
1,SARIMA_manual,test,61.937628,117.060029,109.005144,statsforecast,22.719304,24.308425,10.522410,68.307142,38.016188,5.718792,60.920095
2,WindowAverage,test,99.925490,299.540207,115.617042,baseline,NaN,-22.115069,5.094990,18.902418,0.000000,0.000000,0.000000
3,SeasonalNaive,test,81.828959,369.357753,121.823961,baseline,NaN,0.000000,0.000000,0.000000,18.110025,-5.368515,-23.308239
4,Theta_208,test,111.825577,350.364690,123.026029,statsforecast,22.719304,-36.657704,-0.986726,5.142186,-11.908961,-6.408213,-16.967499
5,Naive,test,111.987330,351.029934,123.140951,baseline,NaN,-36.855377,-1.081060,4.962078,-12.070834,-6.507612,-17.189588
6,random_forest,test,115.317569,387.912911,124.986617,mlforecast,7.561116,-40.925132,-2.596087,-5.023628,-15.403556,-8.103974,-29.502785
7,AutoARIMA_208,test,118.391996,377.178595,127.950773,statsforecast,22.719304,-44.682270,-5.029234,-2.117417,-18.480275,-10.667745,-25.919188


---
## RESUMEN FINAL

Este notebook implementa los ocho modelos obligatorios de la fase 2 y los compara contra los baselines de fase 1 con el mismo split temporal. La conclusion final debe redactarse despues de ejecutar el notebook completo, usando:

- Ranking por MAE, RMSE y MAPE.
- Mejora porcentual contra `SeasonalNaive` y `WindowAverage`.
- Diagnostico de residuos del mejor modelo.
- Comparacion CV vs test para identificar sobreajuste o cambio de regimen.
- Balance entre precision, complejidad computacional e interpretabilidad.

Limitacion metodologica principal: el modelo solo usa historia de casos y calendario. Sin covariables climaticas o entomologicas, los modelos pueden capturar persistencia, pero no necesariamente anticipar con suficiente anticipacion cambios abruptos de regimen epidemiologico.